# Iteris — Week 1: CAMUS Ingestion + Attention Residual U-Net Baseline
**Project:** DRL-based Automated Medical Image Segmentation  
**Dataset:** CAMUS (Cardiac Acquisitions for Multi-structure Ultrasound Segmentation)  
**Target:** Dice ≥ 0.85 on LV Endocardium  

### Setup on Kaggle:
1. Add dataset: `xiaoweixumedicalai/camus` in the Data panel
2. Enable GPU (T4 x2 or P100)
3. Run all cells

### To switch to CHAOS: change `CFG = CAMUS_CFG` → `CFG = CHAOS_CFG`. Nothing else changes.

## 0 · Install & Imports

In [ ]:
%%capture
!pip install monai[all] -q

In [ ]:
import os, json, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import monai
from monai.data import Dataset, CacheDataset
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, Resized,
    ScaleIntensityd, NormalizeIntensityd, ScaleIntensityRanged,
    RandFlipd, RandRotate90d, RandShiftIntensityd,
    RandAffined, RandGaussianNoised,
    AsDiscreted, Activationsd,
)
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.utils import set_determinism
from monai.visualize import blend_images

print('MONAI:', monai.__version__)
print('PyTorch:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1 · Configuration
> Change `CFG = CAMUS_CFG` ↔ `CFG = CHAOS_CFG` to switch datasets. Model and training loop are identical.

In [ ]:
CAMUS_CFG = dict(
    dataset      = 'CAMUS',
    modality     = 'ultrasound',
    data_root    = '/kaggle/input/camus/training',
    image_size   = (256, 256),
    # 0=bg  1=LV_endo  2=LV_epi  3=LA
    num_classes  = 4,
    class_names  = ['background', 'LV_endo', 'LV_epi', 'LA'],
    class_colors = ['#000000', '#00C9A7', '#F59E0B', '#F87171'],
    views        = ['2CH', '4CH'],
    phases       = ['ED', 'ES'],
    spacing      = (1.0, 1.0),   # target pixel spacing (mm)
    normalize    = 'minmax',
    label_frac   = 1.0,
    val_split    = 0.15,
    test_split   = 0.10,
    batch_size   = 8,
    epochs       = 60,
    lr           = 1e-4,
    weight_decay = 1e-5,
    patience     = 10,
    seed         = 42,
    checkpoint   = '/kaggle/working/camus_best.pt',
)

CHAOS_CFG = dict(
    dataset      = 'CHAOS',
    modality     = 'ct',
    data_root    = '/kaggle/input/chaos-dataset',
    image_size   = (256, 256),
    # 0=bg  1=liver  2=r_kidney  3=l_kidney  4=spleen
    num_classes  = 5,
    class_names  = ['background', 'liver', 'right_kidney', 'left_kidney', 'spleen'],
    class_colors = ['#000000', '#818CF8', '#34D399', '#86EFAC', '#FB923C'],
    spacing      = (1.5, 1.5),
    normalize    = 'hu',
    hu_window    = (-17, 201),   # liver window: L=70 W=150 → [70-75, 70+75]
    label_frac   = 1.0,
    val_split    = 0.15,
    test_split   = 0.10,
    batch_size   = 8,
    epochs       = 60,
    lr           = 1e-4,
    weight_decay = 1e-5,
    patience     = 10,
    seed         = 42,
    checkpoint   = '/kaggle/working/chaos_best.pt',
)

# ─── ACTIVE CONFIG ───────────────────────────────────────────────────────────
CFG = CAMUS_CFG
# ─────────────────────────────────────────────────────────────────────────────

set_determinism(CFG['seed'])
print(f"Running: {CFG['dataset']} | {CFG['num_classes']} classes | image {CFG['image_size']}")

## 2 · Ingestion — Build file-list dicts
MONAI transforms expect a list of `{'image': path, 'label': path}` dicts.  
We walk the dataset folder here; all loading and preprocessing is deferred to the transform pipeline.

In [ ]:
def build_camus_dicts(data_root, views, phases):
    """Return [{image: path, label: path, patient, view, phase}, ...]"""
    root    = Path(data_root)
    records = []
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir(): continue
        pid = patient_dir.name
        for view in views:
            for phase in phases:
                img = patient_dir / f'{pid}_{view}_{phase}.mhd'
                lbl = patient_dir / f'{pid}_{view}_{phase}_gt.mhd'
                if img.exists() and lbl.exists():
                    records.append(dict(
                        image   = str(img),
                        label   = str(lbl),
                        patient = pid,
                        view    = view,
                        phase   = phase,
                    ))
    return records


def build_chaos_ct_dicts(data_root):
    """CHAOS CT: DICOM slices + PNG masks → [{image, label}, ...]"""
    import pydicom
    from PIL import Image as PILImage
    LABEL_MAP = {55: 1, 110: 2, 155: 3, 200: 4}
    root    = Path(data_root) / 'CT'
    records = []
    for patient_dir in sorted(root.iterdir()):
        dcm_dir = patient_dir / 'DICOM_anon'
        gt_dir  = patient_dir / 'Ground'
        if not dcm_dir.exists(): continue
        for dcm_f, gt_f in zip(sorted(dcm_dir.glob('*.dcm')), sorted(gt_dir.glob('*.png'))):
            records.append(dict(
                image   = str(dcm_f),
                label   = str(gt_f),
                patient = patient_dir.name,
            ))
    return records


# Build dicts for active dataset
if CFG['dataset'] == 'CAMUS':
    all_dicts = build_camus_dicts(CFG['data_root'], CFG['views'], CFG['phases'])
elif CFG['dataset'] == 'CHAOS':
    all_dicts = build_chaos_ct_dicts(CFG['data_root'])

print(f'Total file pairs: {len(all_dicts)}')
print('Example entry:', {k: v for k, v in all_dicts[0].items() if k in ('image','label','patient')})

## 3 · MONAI Transform Pipelines
All preprocessing — loading, orientation, resampling, normalisation, augmentation —  
is expressed as composable MONAI transforms. Swap `normalize` in CFG for a different modality.

In [ ]:
def build_intensity_transform(cfg):
    """Return the correct intensity normalisation transform for the active modality."""
    if cfg['normalize'] == 'minmax':
        # Ultrasound / greyscale: scale [img_min, img_max] → [0, 1] per image
        return ScaleIntensityd(keys=['image'], minv=0.0, maxv=1.0)
    if cfg['normalize'] == 'zscore':
        # MRI: zero-mean unit-variance
        return NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True)
    if cfg['normalize'] == 'hu':
        # CT: clip to HU window then scale to [0, 1]
        a_min, a_max = cfg['hu_window']
        return ScaleIntensityRanged(
            keys=['image'], a_min=a_min, a_max=a_max,
            b_min=0.0, b_max=1.0, clip=True,
        )
    raise ValueError(f"Unknown normalize: {cfg['normalize']}")


def make_transforms(cfg, split='train'):
    base = [
        LoadImaged(keys=['image', 'label']),
        EnsureChannelFirstd(keys=['image', 'label']),
        # Consistent axial orientation (no-op for 2-D .mhd, safe for 3-D NIfTI)
        Orientationd(keys=['image', 'label'], axcodes='RAS'),
        # Resample to uniform pixel spacing
        Spacingd(
            keys=['image', 'label'],
            pixdim=(*cfg['spacing'], -1),
            mode=('bilinear', 'nearest'),
        ),
        # Resize to fixed spatial dims
        Resized(
            keys=['image', 'label'],
            spatial_size=cfg['image_size'],
            mode=('bilinear', 'nearest'),
        ),
        build_intensity_transform(cfg),
        EnsureTyped(keys=['image', 'label'], dtype=(torch.float32, torch.long)),
    ]

    aug = [
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
        RandRotate90d(keys=['image', 'label'], prob=0.3, max_k=3),
        RandAffined(
            keys=['image', 'label'],
            prob=0.6,
            rotate_range=(0.26,),    # ±15°
            scale_range=(0.1,),
            translate_range=(10,),
            mode=('bilinear', 'nearest'),
            padding_mode='border',
        ),
        RandShiftIntensityd(keys=['image'], offsets=0.15, prob=0.4),
        RandGaussianNoised(keys=['image'], prob=0.3, std=0.02),
    ] if split == 'train' else []

    return Compose(base + aug)


train_tfm = make_transforms(CFG, split='train')
eval_tfm  = make_transforms(CFG, split='val')
print('Transform pipelines built.')

## 4 · Patient-level Split + DataLoaders

In [ ]:
# ─── Label-fraction sub-sampling (for few-shot ablations in Week 4) ───────────
if CFG['label_frac'] < 1.0:
    n = max(1, int(len(all_dicts) * CFG['label_frac']))
    all_dicts = random.sample(all_dicts, n)
    print(f'Using {n} samples ({CFG["label_frac"]*100:.0f}%)')

# Split by patient ID — prevents view/phase leakage across splits
patients = list({d['patient'] for d in all_dicts})
random.shuffle(patients)
n_test  = max(1, int(len(patients) * CFG['test_split']))
n_val   = max(1, int(len(patients) * CFG['val_split']))
test_pts  = set(patients[:n_test])
val_pts   = set(patients[n_test : n_test + n_val])
train_pts = set(patients[n_test + n_val:])

train_dicts = [d for d in all_dicts if d['patient'] in train_pts]
val_dicts   = [d for d in all_dicts if d['patient'] in val_pts]
test_dicts  = [d for d in all_dicts if d['patient'] in test_pts]
print(f'Train: {len(train_dicts)}  Val: {len(val_dicts)}  Test: {len(test_dicts)}')

# CacheDataset pre-loads transformed data into RAM — much faster on Kaggle
train_ds = CacheDataset(train_dicts, transform=train_tfm, cache_rate=1.0, num_workers=2)
val_ds   = CacheDataset(val_dicts,   transform=eval_tfm,  cache_rate=1.0, num_workers=2)
test_ds  = Dataset(test_dicts,       transform=eval_tfm)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

batch = next(iter(train_dl))
print(f'Image batch: {batch["image"].shape}  Label batch: {batch["label"].shape}')

## 5 · EDA — Sanity Check

In [ ]:
from matplotlib.colors import ListedColormap
cmap = ListedColormap(CFG['class_colors'])

fig, axes = plt.subplots(4, 2, figsize=(8, 16))
for row in range(4):
    img  = batch['image'][row, 0].numpy()   # (H, W)
    mask = batch['label'][row, 0].numpy()   # (H, W)
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title('Image'); axes[row, 0].axis('off')
    axes[row, 1].imshow(img, cmap='gray')
    axes[row, 1].imshow(mask, cmap=cmap, alpha=0.5, vmin=0, vmax=CFG['num_classes']-1)
    axes[row, 1].set_title('Overlay'); axes[row, 1].axis('off')

patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CFG['class_colors'][1:], CFG['class_names'][1:])]
fig.legend(handles=patches, loc='upper right')
plt.suptitle(f'{CFG["dataset"]} — Train batch sample')
plt.tight_layout(); plt.show()

## 6 · Model — Attention Residual U-Net
Custom implementation (required for paper — not a wrapper).  
Only `in_channels` and `num_classes` change between datasets.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.skip  = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )
    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.skip(x))


class AttentionGate(nn.Module):
    """Soft attention gate — Oktay et al. (2018)."""
    def __init__(self, F_g, F_x, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g,   F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_x,   F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1,     1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g_up  = F.interpolate(self.W_g(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        alpha = self.psi(self.relu(g_up + self.W_x(x)))
        return x * alpha


class AttentionResUNet(nn.Module):
    """
    Attention Residual U-Net — dataset-agnostic.
    in_channels : 1 greyscale (CAMUS/CHAOS/ACDC/DRIVE/ISIC), 4 for BraTS
    num_classes : from CFG['num_classes']
    """
    def __init__(self, in_channels=1, num_classes=4, features=(64, 128, 256, 512)):
        super().__init__()
        F = features
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ResBlock(in_channels, F[0])
        self.enc2 = ResBlock(F[0], F[1])
        self.enc3 = ResBlock(F[1], F[2])
        self.enc4 = ResBlock(F[2], F[3])
        self.bottleneck = ResBlock(F[3], F[3]*2)

        self.att4 = AttentionGate(F[3]*2, F[3], F[3]//2)
        self.att3 = AttentionGate(F[3],   F[2], F[2]//2)
        self.att2 = AttentionGate(F[2],   F[1], F[1]//2)
        self.att1 = AttentionGate(F[1],   F[0], F[0]//2)

        self.up4  = nn.ConvTranspose2d(F[3]*2, F[3], 2, stride=2)
        self.dec4 = ResBlock(F[3]*2, F[3])
        self.up3  = nn.ConvTranspose2d(F[3], F[2], 2, stride=2)
        self.dec3 = ResBlock(F[2]*2, F[2])
        self.up2  = nn.ConvTranspose2d(F[2], F[1], 2, stride=2)
        self.dec2 = ResBlock(F[1]*2, F[1])
        self.up1  = nn.ConvTranspose2d(F[1], F[0], 2, stride=2)
        self.dec1 = ResBlock(F[0]*2, F[0])

        self.head = nn.Conv2d(F[0], num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b),  self.att4(b,  e4)], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), self.att3(d4, e3)], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), self.att2(d3, e2)], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), self.att1(d2, e1)], 1))
        return self.head(d1)


model = AttentionResUNet(in_channels=1, num_classes=CFG['num_classes']).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params/1e6:.2f}M')

# Dry run
with torch.no_grad():
    sz  = CFG['image_size']
    out = model(torch.zeros(2, 1, sz[0], sz[1]).to(DEVICE))
print(f'Output: {out.shape}  (expect [2, {CFG["num_classes"]}, {sz[0]}, {sz[1]}])')

## 7 · Loss & Metrics (MONAI)

In [ ]:
# DiceCELoss: softmax + Dice + CrossEntropy combined, background excluded
criterion = DiceCELoss(
    to_onehot_y = True,
    softmax     = True,
    include_background = False,
    lambda_dice = 0.5,
    lambda_ce   = 0.5,
)

# DiceMetric: per-class Dice, background excluded, reduction='mean_batch'
dice_metric = DiceMetric(
    include_background = False,
    reduction          = 'mean_batch',   # per-class means across the batch
    get_not_nans       = True,
)

# Post-processing: argmax logits → one-hot for DiceMetric
post_pred  = Compose([Activationsd(keys=['pred'],  softmax=True),
                      AsDiscreted(keys=['pred'],  argmax=True, to_onehot=CFG['num_classes'])])
post_label = AsDiscreted(keys=['label'], to_onehot=CFG['num_classes'])

print('MONAI loss and metric ready.')

## 8 · Training Loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        imgs   = batch['image'].to(device)   # (B, 1, H, W)
        labels = batch['label'].to(device)   # (B, 1, H, W)  integer
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, dice_metric):
    model.eval()
    total_loss = 0.0
    dice_metric.reset()

    for batch in loader:
        imgs   = batch['image'].to(device)
        labels = batch['label'].to(device)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()

        # One-hot predictions and labels for DiceMetric
        preds_oh  = F.one_hot(logits.argmax(1), CFG['num_classes']).permute(0,3,1,2).float()
        labels_oh = F.one_hot(labels.squeeze(1).long(), CFG['num_classes']).permute(0,3,1,2).float()
        dice_metric(y_pred=preds_oh, y=labels_oh)

    per_class_dice, not_nans = dice_metric.aggregate()  # (num_classes-1,)
    dice_metric.reset()
    return total_loss / len(loader), per_class_dice.cpu().numpy()


# ─── Run ─────────────────────────────────────────────────────────────────────
best_dice   = 0.0
patience_ct = 0
history     = {'train_loss': [], 'val_loss': [], 'val_dice_mean': []}

print(f'Training {CFG["dataset"]} | {CFG["epochs"]} epochs | {DEVICE}')
for epoch in range(1, CFG['epochs'] + 1):
    tr_loss                  = train_epoch(model, train_dl, optimizer, criterion, DEVICE)
    vl_loss, per_class_dice  = eval_epoch(model, val_dl, criterion, DEVICE, dice_metric)
    scheduler.step()

    mean_dice = float(np.nanmean(per_class_dice))
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_dice_mean'].append(mean_dice)

    if mean_dice > best_dice:
        best_dice   = mean_dice
        patience_ct = 0
        torch.save(model.state_dict(), CFG['checkpoint'])
        marker = '✓'
    else:
        patience_ct += 1
        marker = ' '

    class_str = '  '.join(f'{CFG["class_names"][c+1]}:{per_class_dice[c]:.3f}'
                          for c in range(len(per_class_dice)))
    print(f'Ep {epoch:03d} | tr {tr_loss:.4f} | vl {vl_loss:.4f} | '
          f'Dice {mean_dice:.4f} {marker} | {class_str}')

    if patience_ct >= CFG['patience']:
        print(f'Early stop at epoch {epoch}.')
        break

print(f'Best val Dice: {best_dice:.4f}')

## 9 · Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss curves'); ax1.legend()

ax2.plot(history['val_dice_mean'], label='Mean Dice')
ax2.axhline(0.85, color='red', linestyle='--', label='Target 0.85')
ax2.set(xlabel='Epoch', ylabel='Dice', title=f'{CFG["dataset"]} — Val Dice'); ax2.legend()

plt.suptitle(f'{CFG["dataset"]} Attention Residual U-Net — Week 1 Baseline')
plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=150)
plt.show()

## 10 · Test-set Evaluation

In [ ]:
model.load_state_dict(torch.load(CFG['checkpoint'], map_location=DEVICE))
_, test_dice = eval_epoch(model, test_dl, criterion, DEVICE, dice_metric)

print('\n── Test Results ────────────────────────────────')
for c, name in enumerate(CFG['class_names'][1:]):
    print(f'  {name:15s}: Dice {test_dice[c]:.4f}')
print(f'  Mean Dice      : {np.nanmean(test_dice):.4f}')
print('────────────────────────────────────────────────')
print(f'  LV_endo target (≥0.85): {"PASS ✓" if test_dice[0] >= 0.85 else "not yet"}')

## 11 · Qualitative Visualisation

In [ ]:
model.eval()
cmap    = ListedColormap(CFG['class_colors'])
n_show  = 4
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4*n_show))

for row, batch in enumerate(test_dl):
    if row >= n_show: break
    img   = batch['image'][0].to(DEVICE)
    label = batch['label'][0, 0].numpy()
    with torch.no_grad():
        pred = model(img.unsqueeze(0)).argmax(1)[0].cpu().numpy()
    img_np = img[0].cpu().numpy()

    axes[row,0].imshow(img_np, cmap='gray'); axes[row,0].set_title('Input');       axes[row,0].axis('off')
    axes[row,1].imshow(img_np, cmap='gray')
    axes[row,1].imshow(label, cmap=cmap, alpha=0.5, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
    axes[row,2].imshow(img_np, cmap='gray')
    axes[row,2].imshow(pred, cmap=cmap, alpha=0.5, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,2].set_title('Prediction');   axes[row,2].axis('off')

patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CFG['class_colors'][1:], CFG['class_names'][1:])]
fig.legend(handles=patches, loc='upper right')
plt.suptitle(f'{CFG["dataset"]} — Qualitative Results')
plt.tight_layout()
plt.savefig('/kaggle/working/qualitative_results.png', dpi=150)
plt.show()

## 12 · Save Summary JSON
Checkpoint info consumed by the Week 2 DRL pipeline.

In [ ]:
summary = dict(
    dataset        = CFG['dataset'],
    model          = 'AttentionResUNet',
    num_classes    = CFG['num_classes'],
    image_size     = CFG['image_size'],
    checkpoint     = CFG['checkpoint'],
    best_val_dice  = round(float(best_dice), 4),
    test_per_class = {CFG['class_names'][c+1]: round(float(test_dice[c]), 4)
                      for c in range(len(test_dice))},
    test_dice_mean = round(float(np.nanmean(test_dice)), 4),
)
out = CFG['checkpoint'].replace('.pt', '_summary.json')
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))